# Peer Study Group Finder
## Assignment 3: End-to-End Portal Design

| Item | Detail |
|---|---|
| Team member 1 | [Maham Waqas], [25l-5225] |
| Team member 2 | [Khadija Amir], [25l-5110] |
| Course / Institution | [Programming for Business], FAST - National University, Lahore |
|

### Overview
Peer Study Group Finder is a console-based portal written in Python. Students register a profile (subjects, grades, interest level, availability, mentoring details), find study groups for their subjects, and negotiate a **time, location and price** through a propose, counter and accept process. A group is confirmed only when every member has accepted the current proposal. All data is stored in two JSON files (`users.json`, `groups.json`).

### Concepts demonstrated
| Requirement | Implementation |
|---|---|
| Modular functions (`def`) | One documented function per feature, with parameters and return values |
| Loops (`for`, `while`) | `while True` main menu loop; `for` loops to search, filter, display and total records |
| File handling | JSON files read at start-up and saved after every change |
| Exception handling | `FileNotFoundError`, `JSONDecodeError`, `PermissionError`/`OSError`, `ValueError`, `KeyboardInterrupt`/`EOFError`, and `finally` |

## 1. Set-up and constants
Imports, file names and the fixed lists used for validation. The school of a user is looked up from `PROGRAMS` instead of being stored in every record.

In [3]:
import json   # built-in module for reading/writing JSON files
import os      # lets us check/delete files (used in the demo reset)

# File names where the portal keeps its data
USERS_FILE = "users.json"
GROUPS_FILE = "groups.json"

# programme -> school (the school is looked up, never stored in a user record)
PROGRAMS = {
    "BSBA": "School of Management Sciences",
    "BSAF": "School of Management Sciences",
    "BSFT": "School of Management Sciences",
    "BBA ": "Scholl of Management Sciences",
    "BSCS": "School of Computing",
    "BSAI": "School of Computing",
    "BSDS": "School of Computing",
    "BSSE": "School of Computing",
    "BSCY": "School of Computing",
    "BSCE": "School of Engineering",
    "BSEE": "School of Engineering",
    "BSCompE": "School of Engineering"
    
}

GRADES = ["A+", "A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]
INTERESTS = ["intrigued", "marks", "both"]   # genuinely interested / only for grades / both
OPEN_STATUSES = ["proposed", "countered"]    # a group still being negotiated

print("Set-up done. Programmes available:", list(PROGRAMS.keys()))

Set-up done. Programmes available: ['BSBA', 'BSAF', 'BSFT', 'BBA ', 'BSCS', 'BSAI', 'BSDS', 'BSSE', 'BSCY', 'BSCE', 'BSEE', 'BSCompE']


## 2. File handling: `load_data()` and `save_data()`
`load_data` returns an empty list when the file is missing or damaged, so the first run works. `save_data` returns `True` on success and `False` (with a warning) if the file cannot be written.

In [4]:
def load_data(filename):
    """Read a JSON file and return its contents as a list.

    Parameters:
        filename (str): path of the JSON file.
    Returns:
        list: the stored records, or an empty list if the file is
              missing, damaged or unreadable (so the first run still works).
    """
    try:
        with open(filename, "r") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        print(f"  ! {filename} does not hold a list. Starting empty.")
        return []
    except FileNotFoundError:
        return []                      # first run: no file yet, that's fine
    except json.JSONDecodeError:
        print(f"  ! {filename} is damaged. Starting with an empty list.")
        return []
    except PermissionError:
        print(f"  ! No permission to read {filename}. Starting empty.")
        return []


def save_data(filename, data):
    """Write a list of records to a JSON file.

    Parameters:
        filename (str): path of the JSON file.
        data (list): the records to store.
    Returns:
        bool: True if saved, False if something went wrong.
    """
    try:
        with open(filename, "w") as f:
            json.dump(data, f, indent=4)   # indent=4 makes the file readable
        return True
    except (PermissionError, OSError) as error:
        print(f"  ! Could not save {filename}: {error}")
        return False

**Test: file error handling**  
Missing file, damaged file, unwritable path, and a normal save/load.

In [5]:
print("1) Loading a file that does not exist:")
print("   result =", load_data("this_file_does_not_exist.json"))

print("\n2) Loading a damaged file:")
with open("broken_test.json", "w") as f:
    f.write("{ this is not valid json")
print("   result =", load_data("broken_test.json"))
os.remove("broken_test.json")          # clean up our test file

print("\n3) Saving to a folder that does not exist:")
print("   result =", save_data("no_such_folder/x.json", []))

print("\n4) Normal save and load:")
print("   saved  =", save_data("test_ok.json", [{"hello": "world"}]))
print("   loaded =", load_data("test_ok.json"))
os.remove("test_ok.json")

1) Loading a file that does not exist:
   result = []

2) Loading a damaged file:
  ! broken_test.json is damaged. Starting with an empty list.
   result = []

3) Saving to a folder that does not exist:
  ! Could not save no_such_folder/x.json: [Errno 44] No such file or directory: 'no_such_folder/x.json'
   result = False

4) Normal save and load:
   saved  = True
   loaded = [{'hello': 'world'}]


## 3. Input-validation helpers
Each helper repeats its question until the answer is valid, so invalid input never reaches the data (`ValueError` is handled inside).

In [6]:
def get_valid_int(prompt, low, high):
    """Keep asking until the user enters a whole number between low and high."""
    while True:
        try:
            value = int(input(prompt).strip())
        except ValueError:
            print("  ! Please type a whole number, like 3.")
            continue                       # jump back to the top of the loop
        if low <= value <= high:
            return value
        print(f"  ! The number must be between {low} and {high}.")


def get_valid_float(prompt, low, high):
    """Keep asking until the user enters a decimal number between low and high."""
    while True:
        try:
            value = float(input(prompt).strip())
        except ValueError:
            print("  ! Please type a number, like 3.5.")
            continue
        if low <= value <= high:
            return value
        print(f"  ! The number must be between {low} and {high}.")


def get_choice(prompt, options):
    """Keep asking until the answer matches one of the options (ignoring upper/lower case).

    Returns the option exactly as it is written in the options list.
    """
    while True:
        answer = input(prompt).strip().lower()
        for option in options:             # look through the allowed options
            if answer == option.lower():
                return option
        print("  ! Please choose one of:", ", ".join(options))


def get_text(prompt):
    """Keep asking until the user types something that is not empty."""
    while True:
        answer = input(prompt).strip()
        if answer != "":
            return answer
        print("  ! This cannot be empty.")

## 4. `add_record()` and `search_record()`
General-purpose functions used for both user and group records.

In [7]:
def add_record(records, new_record):
    """Append a user or group record to its list and return the list."""
    records.append(new_record)
    return records


def search_record(records, key, value):
    """Return every record where record[key] equals value.

    Example: search_record(users, "user_id", "ali123") -> [ali's record]
    An empty list means nothing matched.
    """
    results = []
    for record in records:
        if record.get(key) == value:
            results.append(record)
    return results

In [8]:
# Quick test with made-up data
demo = []
add_record(demo, {"user_id": "a1", "name": "Test A"})
add_record(demo, {"user_id": "b2", "name": "Test B"})
print(search_record(demo, "user_id", "b2"))     # one match
print(search_record(demo, "user_id", "zzz"))    # no match -> []

[{'user_id': 'b2', 'name': 'Test B'}]
[]


## 5. Registration and login
`register_user()` collects and validates a full profile; `user_login()` allows three attempts and returns the user record or `None`.

In [9]:
def register_user(users):
    """Collect a full, validated profile, add it to users, and return it."""
    print("\n--- Register a new account ---")

    # 1) user_id: must be unique and have no spaces
    while True:
        user_id = get_text("Choose a user ID (no spaces): ")
        if " " in user_id:
            print("  ! No spaces allowed.")
        elif search_record(users, "user_id", user_id):
            print("  ! That ID is already taken. Try another.")
        else:
            break

    # 2) password: at least 4 characters
    while True:
        password = get_text("Choose a password (min 4 characters): ")
        if len(password) >= 4:
            break
        print("  ! Too short.")

    name = get_text("Full name: ")
    gender = get_choice("Gender (M/F): ", ["M", "F"])
    cgpa = get_valid_float("CGPA (0.0 - 4.0): ", 0.0, 4.0)
    program = get_choice("Programme (" + "/".join(PROGRAMS) + "): ", list(PROGRAMS))
    semester = get_valid_int("Semester (1-8): ", 1, 8)

    # 3) subjects: keep adding until the user says no
    subjects = []
    while True:
        subject_name = get_text("Subject name: ")
        grade = get_choice("Grade (" + ", ".join(GRADES) + "): ", GRADES)
        interest = get_choice("Interest (intrigued / marks / both): ", INTERESTS)
        subjects.append({"name": subject_name, "grade": grade, "interest": interest})
        if get_choice("Add another subject? (y/n): ", ["y", "n"]) == "n":
            break

    # 4) availability slots
    availability = []
    while True:
        availability.append(get_text("Available time slot (e.g. Mon 4-6pm): "))
        if get_choice("Add another slot? (y/n): ", ["y", "n"]) == "n":
            break

    # 5) mentor details (only asked if the user wants to mentor)
    can_mentor = get_choice("Can you mentor others? (y/n): ", ["y", "n"]) == "y"
    charges, proof = 0.0, ""
    if can_mentor:
        charges = get_valid_float("Your charge per person in Rs (0 = free): ", 0, 100000)
        proof = get_text("Proof of knowledge (project, midterm result, previous record): ")

    new_user = {
        "user_id": user_id, "password": password, "name": name, "gender": gender,
        "cgpa": cgpa, "program": program, "semester": semester,
        "subjects": subjects, "availability": availability,
        "can_mentor": can_mentor, "charges": charges, "proof": proof,
    }
    add_record(users, new_user)
    print(f"  Account created. Welcome, {name}!")
    return new_user


def user_login(users):
    """Check user ID and password. Allows three attempts.

    Returns the user's record (dict) on success, or None on failure.
    """
    print("\n--- Login ---")
    for attempt in range(3):
        user_id = input("User ID: ").strip()
        password = input("Password: ").strip()
        matches = search_record(users, "user_id", user_id)
        if matches and matches[0]["password"] == password:
            print(f"  Welcome back, {matches[0]['name']}!")
            return matches[0]
        print(f"  ! Wrong ID or password. Attempts left: {2 - attempt}")
    print("  ! Too many failed attempts.")
    return None

## 6. Study groups and negotiation
**Negotiation rules**
- Creating a group records the first proposal; the creator is placed in `accepted_by` and the status is `proposed`.
- A **counter** changes the time, location or price, adds an entry to `history`, sets the status to `countered` and resets `accepted_by` to the member who countered.
- An **accept** adds the member to `accepted_by`. When `accepted_by` contains every member, the status becomes `confirmed`.
- A new member joining resets acceptance, so everyone agrees to the proposal again.

In [10]:
def generate_group_id(groups):
    """Return the next free group ID such as 'G001', 'G002'..."""
    highest = 0
    for group in groups:
        number = int(group["group_id"][1:])      # 'G012' -> '012' -> 12
        if number > highest:
            highest = number
    return f"G{highest + 1:03d}"


def create_group(user, groups):
    """Build a new study group with the creator's first proposal.

    Returns the new group (dict).
    """
    print("\n--- Create a study group ---")
    subject = get_text("Subject for the group: ")
    max_members = get_valid_int("Maximum members (2-8): ", 2, 8)

    mentor_id = None
    if user["can_mentor"]:
        print(f"  (Your usual mentor charge is Rs {user['charges']:.0f})")
        if get_choice("Will you mentor this group? (y/n): ", ["y", "n"]) == "y":
            mentor_id = user["user_id"]

    time = get_text("Proposed time (e.g. Mon 4-6pm): ")
    location = get_text("Proposed location: ")
    price = get_valid_float("Price per person in Rs (0 = free): ", 0, 100000)

    group = {
        "group_id": generate_group_id(groups),
        "subject": subject,
        "creator_id": user["user_id"],
        "members": [user["user_id"]],
        "max_members": max_members,
        "mentor_id": mentor_id,
        "time": time, "location": location, "price": price,
        "status": "proposed",
        "accepted_by": [user["user_id"]],       # creator agrees with own proposal
        "history": [{"by": user["user_id"], "time": time,
                     "location": location, "price": price}],
    }
    add_record(groups, group)
    print(f"  Group {group['group_id']} created and waiting for members.")
    return group

In [11]:
def find_matches(user, groups):
    """Return open groups that share a subject with the user."""
    my_subjects = []
    for s in user["subjects"]:
        my_subjects.append(s["name"].lower())

    matches = []
    for group in groups:
        is_open = group["status"] in OPEN_STATUSES
        has_space = len(group["members"]) < group["max_members"]
        not_member = user["user_id"] not in group["members"]
        same_subject = group["subject"].lower() in my_subjects
        if is_open and has_space and not_member and same_subject:
            matches.append(group)
    return matches


def join_group(user, group):
    """Add the user to the group. Returns True if joined, False if not allowed."""
    if user["user_id"] in group["members"]:
        print("  ! You are already in this group.")
        return False
    if len(group["members"]) >= group["max_members"]:
        print("  ! This group is full.")
        return False
    if group["status"] not in OPEN_STATUSES:
        print("  ! This group is no longer open.")
        return False

    group["members"].append(user["user_id"])
    group["accepted_by"] = []          # the group must agree again
    group["status"] = "proposed"
    print(f"  You joined group {group['group_id']}. Everyone must accept the proposal again.")
    return True

In [12]:
def show_group(group, users):
    """Print one group's details in a readable block."""
    names = []
    for member_id in group["members"]:
        found = search_record(users, "user_id", member_id)
        names.append(found[0]["name"] if found else member_id)

    mentor = "none"
    if group["mentor_id"]:
        found = search_record(users, "user_id", group["mentor_id"])
        mentor = found[0]["name"] if found else group["mentor_id"]

    print(f"\n[{group['group_id']}] {group['subject']}  |  status: {group['status'].upper()}")
    print(f"   Members ({len(group['members'])}/{group['max_members']}): {', '.join(names)}")
    print(f"   Mentor: {mentor}")
    print(f"   Current proposal: {group['time']} at {group['location']}, Rs {group['price']:.0f} per person")
    print(f"   Accepted by: {', '.join(group['accepted_by']) or 'nobody yet'}")


def counter_proposal(user, group):
    """Let a member change the time, location or price (press Enter to keep a value)."""
    if group["status"] in ("confirmed", "cancelled"):
        print(f"  ! This group is already {group['status']}.")
        return

    print("  Press Enter to keep the current value.")
    new_time = input(f"  Time [{group['time']}]: ").strip() or group["time"]
    new_location = input(f"  Location [{group['location']}]: ").strip() or group["location"]
    new_price = group["price"]
    if get_choice(f"  Change the price (now Rs {group['price']:.0f})? (y/n): ", ["y", "n"]) == "y":
        new_price = get_valid_float("  New price per person in Rs: ", 0, 100000)

    unchanged = (new_time == group["time"] and new_location == group["location"]
                 and new_price == group["price"])
    if unchanged:
        print("  Nothing changed, so there is nothing to counter.")
        return

    group["time"], group["location"], group["price"] = new_time, new_location, new_price
    group["history"].append({"by": user["user_id"], "time": new_time,
                             "location": new_location, "price": new_price})
    group["status"] = "countered"
    group["accepted_by"] = [user["user_id"]]      # only the person who countered agrees
    print("  Counter-offer sent. The others must accept it.")


def accept_proposal(user, group):
    """Accept the current proposal. Confirms the group when everyone has accepted."""
    if group["status"] == "confirmed":
        print("  This group is already confirmed.")
        return
    if group["status"] == "cancelled":
        print("  ! This group was cancelled.")
        return
    if user["user_id"] in group["accepted_by"]:
        print("  You have already accepted this proposal.")
        return

    group["accepted_by"].append(user["user_id"])

    everyone_agreed = True
    for member_id in group["members"]:
        if member_id not in group["accepted_by"]:
            everyone_agreed = False

    if everyone_agreed and len(group["members"]) >= 2:
        group["status"] = "confirmed"
        print("  All members accepted. The group is now CONFIRMED!")
    else:
        print("  Accepted. Waiting for the other members.")

## 7. Report generation
Groups per subject, confirmed vs open groups, number of mentors, average mentor charge and average CGPA.

In [13]:
def generate_report(users, groups):
    """Print: groups per subject, confirmed vs open groups, mentor numbers, average CGPA."""
    print("\n========== PORTAL REPORT ==========")
    print(f"Registered students: {len(users)}")
    print(f"Study groups:        {len(groups)}")

    per_subject = {}
    confirmed, still_open = 0, 0
    for group in groups:
        subject = group["subject"].title()
        per_subject[subject] = per_subject.get(subject, 0) + 1
        if group["status"] == "confirmed":
            confirmed += 1
        elif group["status"] in OPEN_STATUSES:
            still_open += 1

    print(f"Confirmed groups:    {confirmed}")
    print(f"Open groups:         {still_open}")
    print("Groups per subject:")
    for subject, count in per_subject.items():
        print(f"   - {subject}: {count}")

    mentor_count, charge_total, cgpa_total = 0, 0.0, 0.0
    for user in users:
        cgpa_total += user["cgpa"]
        if user["can_mentor"]:
            mentor_count += 1
            charge_total += user["charges"]

    avg_charge = charge_total / mentor_count if mentor_count else 0
    avg_cgpa = cgpa_total / len(users) if users else 0
    print(f"Mentors:             {mentor_count}")
    print(f"Average mentor charge: Rs {avg_charge:.0f}")
    print(f"Average CGPA:        {avg_cgpa:.2f}")
    print("===================================")

## 8. Menus and `main()`
`main()` loads both files, runs the `while True` menu loop until the user exits, and uses `try / except / finally` so data is always saved.

In [14]:
def save_all(users, groups):
    """Save both data files."""
    save_data(USERS_FILE, users)
    save_data(GROUPS_FILE, groups)


def show_profile(user):
    """Print the user's own profile."""
    print("\n--- My profile ---")
    print(f"Name: {user['name']}  ({user['user_id']})")
    print(f"Programme: {user['program']} ({PROGRAMS[user['program']]}), semester {user['semester']}")
    print(f"CGPA: {user['cgpa']}")
    print("Subjects:")
    for s in user["subjects"]:
        print(f"   - {s['name']}: grade {s['grade']}, interest: {s['interest']}")
    print("Availability:", ", ".join(user["availability"]))
    if user["can_mentor"]:
        charge = "free" if user["charges"] == 0 else f"Rs {user['charges']:.0f}"
        print(f"Mentor: yes ({charge}). Proof: {user['proof']}")
    else:
        print("Mentor: no")


def edit_availability(user):
    """Add to, or replace, the user's availability slots."""
    print("Current slots:", ", ".join(user["availability"]))
    mode = get_choice("Type 'add' to add slots or 'replace' to start over: ", ["add", "replace"])
    slots = [] if mode == "replace" else user["availability"]
    while True:
        slots.append(get_text("New time slot: "))
        if get_choice("Add another slot? (y/n): ", ["y", "n"]) == "n":
            break
    user["availability"] = slots
    print("  Availability updated.")


def my_groups_menu(user, users, groups):
    """List the groups this user belongs to and let them counter or accept."""
    mine = []
    for group in groups:
        if user["user_id"] in group["members"]:
            mine.append(group)

    if not mine:
        print("\n  You are not in any group yet.")
        return

    for group in mine:
        show_group(group, users)

    pick = get_valid_int("\nEnter the position of a group to open (1 = first, 0 = back): ", 0, len(mine))
    if pick == 0:
        return
    group = mine[pick - 1]
    action = get_valid_int("1 = Counter-offer, 2 = Accept, 3 = Back: ", 1, 3)
    if action == 1:
        counter_proposal(user, group)
    elif action == 2:
        accept_proposal(user, group)


def user_menu(user, users, groups):
    """The menu shown after login. Returns when the user logs out."""
    while True:
        print(f"\n===== Menu ({user['name']}) =====")
        print("1 View my profile       2 Edit my availability")
        print("3 Create a study group  4 Find matching groups and join")
        print("5 My groups (counter / accept)")
        print("6 Reports               7 Logout")
        choice = get_valid_int("Choose (1-7): ", 1, 7)

        if choice == 1:
            show_profile(user)
        elif choice == 2:
            edit_availability(user)
        elif choice == 3:
            create_group(user, groups)
        elif choice == 4:
            matches = find_matches(user, groups)
            if not matches:
                print("\n  No matching open groups right now.")
            else:
                for group in matches:
                    show_group(group, users)
                pick = get_valid_int("\nEnter the position of the group to join (0 = cancel): ", 0, len(matches))
                if pick != 0:
                    join_group(user, matches[pick - 1])
        elif choice == 5:
            my_groups_menu(user, users, groups)
        elif choice == 6:
            generate_report(users, groups)
        else:
            print("  Logged out.")
            return

        save_all(users, groups)       # save after every action


def main():
    """Run the portal until the user chooses Exit."""
    users = load_data(USERS_FILE)     # read existing records at start-up
    groups = load_data(GROUPS_FILE)
    print("=" * 40)
    print("   PEER STUDY GROUP FINDER")
    print("=" * 40)

    try:
        while True:
            print("\n===== Start menu =====")
            print("1 Register    2 Login    3 Exit")
            choice = get_valid_int("Choose (1-3): ", 1, 3)

            if choice == 1:
                register_user(users)
                save_all(users, groups)
            elif choice == 2:
                user = user_login(users)
                if user is not None:
                    user_menu(user, users, groups)
            else:
                print("Exiting the portal...")
                break
    except (KeyboardInterrupt, EOFError):
        print("\nInterrupted. Closing safely...")
    finally:
        save_all(users, groups)      # always runs
        print("Data saved. Goodbye!")

## 9. Sample test runs
`run_scripted()` supplies prepared answers in place of keyboard input so that each run can be repeated. Every question is printed with its answer, as in a live session. The first cell clears old demo data.

In [15]:
import builtins

def run_scripted(answers):
    """Run main() with a prepared list of answers instead of real typing."""
    answer_iter = iter(answers)
    real_input = builtins.input

    def fake_input(prompt=""):
        try:
            answer = next(answer_iter)
        except StopIteration:
            raise EOFError              # no more answers: act like the user left
        print(prompt + answer)          # show question + answer, like a real session
        return answer

    builtins.input = fake_input
    try:
        main()
    finally:
        builtins.input = real_input     # always restore the real input()


# Clean start for the demo: remove old data files if they exist
for file_name in (USERS_FILE, GROUPS_FILE):
    try:
        os.remove(file_name)
        print("Removed old", file_name)
    except FileNotFoundError:
        print(file_name, "did not exist yet (that's fine)")

users.json did not exist yet (that's fine)
groups.json did not exist yet (that's fine)


### Test run 1: Registering three students (with invalid inputs)
Includes a duplicate user ID, a short password, a non-numeric CGPA (`abc`), an out-of-range CGPA (`9`) and a lower-case programme name.

In [16]:
run_scripted([
    # --- Ali (mentor) ---
    "1",
    "ali123", "pass1234", "Ali Khan", "M", "3.6", "BSCS", "3",
    "Python", "A", "both", "y",
    "Statistics", "A-", "marks", "n",
    "Mon 4-6pm", "y", "Wed 5-7pm", "n",
    "y", "500", "Built a library app in Python; midterm 92/100",
    # --- Sara (learner, makes mistakes) ---
    "1",
    "ali123", "sara01",              # duplicate ID, then a free one
    "ab", "sara2026",                # password too short, then fine
    "Sara Ahmed", "F",
    "abc", "9", "3.4",               # not a number, out of range, valid
    "bsba", "1",
    "Python", "B+", "intrigued", "n",
    "Mon 4-6pm", "y", "Fri 2-4pm", "n",
    "n",
    # --- Hamza (learner) ---
    "1",
    "hamza9", "hamza2026", "Hamza Raza", "M", "3.0", "BSCS", "3",
    "Python", "B", "marks", "y", "Accounting", "B-", "both", "n",
    "Mon 4-6pm", "n",
    "n",
    # --- Exit ---
    "3",
])

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 1

--- Register a new account ---
Choose a user ID (no spaces): ali123
Choose a password (min 4 characters): pass1234
Full name: Ali Khan
Gender (M/F): M
CGPA (0.0 - 4.0): 3.6
Programme (BSBA/BSAF/BSFT/BBA /BSCS/BSAI/BSDS/BSSE/BSCY/BSCE/BSEE/BSCompE): BSCS
Semester (1-8): 3
Subject name: Python
Grade (A+, A, A-, B+, B, B-, C+, C, C-, D, F): A
Interest (intrigued / marks / both): both
Add another subject? (y/n): y
Subject name: Statistics
Grade (A+, A, A-, B+, B, B-, C+, C, C-, D, F): A-
Interest (intrigued / marks / both): marks
Add another subject? (y/n): n
Available time slot (e.g. Mon 4-6pm): Mon 4-6pm
Add another slot? (y/n): y
Available time slot (e.g. Mon 4-6pm): Wed 5-7pm
Add another slot? (y/n): n
Can you mentor others? (y/n): y
Your charge per person in Rs (0 = free): 500
Proof of knowledge (project, midterm result, previous record): Built a library app in Python; midterm 92/100
  

**Data file created: `users.json`**

In [17]:
with open(USERS_FILE) as f:
    print(f.read()[:1100], "\n... (file continues)")
print("\nTotal users stored:", len(load_data(USERS_FILE)))

[
    {
        "user_id": "ali123",
        "password": "pass1234",
        "name": "Ali Khan",
        "gender": "M",
        "cgpa": 3.6,
        "program": "BSCS",
        "semester": 3,
        "subjects": [
            {
                "name": "Python",
                "grade": "A",
                "interest": "both"
            },
            {
                "name": "Statistics",
                "grade": "A-",
                "interest": "marks"
            }
        ],
        "availability": [
            "Mon 4-6pm",
            "Wed 5-7pm"
        ],
        "can_mentor": true,
        "charges": 500.0,
        "proof": "Built a library app in Python; midterm 92/100"
    },
    {
        "user_id": "sara01",
        "password": "sara2026",
        "name": "Sara Ahmed",
        "gender": "F",
        "cgpa": 3.4,
        "program": "BSBA",
        "semester": 1,
        "subjects": [
            {
                "name": "Python",
                "grade": "B+",
           

### Test run 2: Login (one wrong password) and group creation

In [18]:
run_scripted([
    "2",
    "ali123", "wrongpw",             # failed attempt 1
    "ali123", "pass1234",            # success on attempt 2
    "1",                             # view profile
    "3",                             # create group
    "Python", "4", "y",              # subject, max members, I will mentor
    "Mon 4-6pm", "Library Room 2", "500",
    "7",                             # logout
    "3",                             # exit
])

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login ---
User ID: ali123
Password: wrongpw
  ! Wrong ID or password. Attempts left: 2
User ID: ali123
Password: pass1234
  Welcome back, Ali Khan!

===== Menu (Ali Khan) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 1

--- My profile ---
Name: Ali Khan  (ali123)
Programme: BSCS (School of Computing), semester 3
CGPA: 3.6
Subjects:
   - Python: grade A, interest: both
   - Statistics: grade A-, interest: marks
Availability: Mon 4-6pm, Wed 5-7pm
Mentor: yes (Rs 500). Proof: Built a library app in Python; midterm 92/100

===== Menu (Ali Khan) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 3

--- Create a study

### Test run 3: Matching and joining a group

In [19]:
run_scripted([
    "2", "sara01", "sara2026", "4", "1", "7",       # Sara: find matches, join #1, logout
    "2", "hamza9", "hamza2026", "4", "1", "7",      # Hamza: same
    "3",
])

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login ---
User ID: sara01
Password: sara2026
  Welcome back, Sara Ahmed!

===== Menu (Sara Ahmed) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 4

[G001] Python  |  status: PROPOSED
   Members (1/4): Ali Khan
   Mentor: Ali Khan
   Current proposal: Mon 4-6pm at Library Room 2, Rs 500 per person
   Accepted by: ali123

Enter the position of the group to join (0 = cancel): 1
  You joined group G001. Everyone must accept the proposal again.

===== Menu (Sara Ahmed) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 7
  Logged out.

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login -

### Test run 4: Negotiation (counter, accept, accept) ending in `confirmed`

In [20]:
run_scripted([
    # Sara counters: keep time (Enter), new location, change price
    "2", "sara01", "sara2026", "5", "1", "1", "", "Cafeteria", "y", "300", "7",
    # Ali accepts
    "2", "ali123", "pass1234", "5", "1", "2", "7",
    # Hamza accepts -> confirmed
    "2", "hamza9", "hamza2026", "5", "1", "2", "7",
    "3",
])

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login ---
User ID: sara01
Password: sara2026
  Welcome back, Sara Ahmed!

===== Menu (Sara Ahmed) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 5

[G001] Python  |  status: PROPOSED
   Members (3/4): Ali Khan, Sara Ahmed, Hamza Raza
   Mentor: Ali Khan
   Current proposal: Mon 4-6pm at Library Room 2, Rs 500 per person
   Accepted by: nobody yet

Enter the position of a group to open (1 = first, 0 = back): 1
1 = Counter-offer, 2 = Accept, 3 = Back: 1
  Press Enter to keep the current value.
  Time [Mon 4-6pm]: 
  Location [Library Room 2]: Cafeteria
  Change the price (now Rs 500)? (y/n): y
  New price per person in Rs: 300
  Counter-offer sent. The others must accept it.

===== Menu (Sara Ahmed) =====
1 View my profile       2 Edit my availabil

### Test run 5: Report, menu input errors and accepting an already confirmed group

In [21]:
run_scripted([
    "2", "ali123", "pass1234",
    "6",                          # report
    "abc", "99", "5",             # letters, out of range, then open My groups
    "1", "2",                     # open group 1 -> accept (already confirmed)
    "7",
    "3",
])

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login ---
User ID: ali123
Password: pass1234
  Welcome back, Ali Khan!

===== Menu (Ali Khan) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): 6

========== PORTAL REPORT ==========
Registered students: 3
Study groups:        1
Confirmed groups:    1
Open groups:         0
Groups per subject:
   - Python: 1
Mentors:             1
Average mentor charge: Rs 500
Average CGPA:        3.33

===== Menu (Ali Khan) =====
1 View my profile       2 Edit my availability
3 Create a study group  4 Find matching groups and join
5 My groups (counter / accept)
6 Reports               7 Logout
Choose (1-7): abc
  ! Please type a whole number, like 3.
Choose (1-7): 99
  ! The number must be between 1 and 7.
Choose (1-7): 5

[G001] Python  |  status: CONFIRMED
   Mem

### Test run 6: Interrupted input
The input ends during login, so the portal closes safely and saves its data.

In [22]:
run_scripted(["2", "sara01"])     # input stops halfway through login

   PEER STUDY GROUP FINDER

===== Start menu =====
1 Register    2 Login    3 Exit
Choose (1-3): 2

--- Login ---
User ID: sara01

Interrupted. Closing safely...
Data saved. Goodbye!


**Data file after the runs: `groups.json`** (including negotiation history)

In [23]:
groups_on_disk = load_data(GROUPS_FILE)
print(json.dumps(groups_on_disk[0], indent=4))

{
    "group_id": "G001",
    "subject": "Python",
    "creator_id": "ali123",
    "members": [
        "ali123",
        "sara01",
        "hamza9"
    ],
    "max_members": 4,
    "mentor_id": "ali123",
    "time": "Mon 4-6pm",
    "location": "Cafeteria",
    "price": 300.0,
    "status": "confirmed",
    "accepted_by": [
        "sara01",
        "ali123",
        "hamza9"
    ],
    "history": [
        {
            "by": "ali123",
            "time": "Mon 4-6pm",
            "location": "Library Room 2",
            "price": 500.0
        },
        {
            "by": "sara01",
            "time": "Mon 4-6pm",
            "location": "Cafeteria",
            "price": 300.0
        }
    ]
}


## 10. Running the portal interactively
Remove the `#` below to start the portal with real keyboard input.

In [28]:
# main()      # <- remove the '#' to run the portal with your own typing

# 11. Limitations
Passwords are stored as plain text (a production system would hash them). Real-time chat, online payments and file uploads are outside the scope of this project.